<a href="https://colab.research.google.com/github/pavelpryadokhin/AI-Assistant/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22%D0%9A%D0%BE%D0%BD%D1%81%D1%83%D0%BB%D1%8C%D1%82%D0%B0%D0%BD%D1%82_RAG%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install chromadb
!pip install llama_index
!pip install llama-index-vector-stores-chroma
!pip install llama-index-embeddings-huggingface
!pip install llama-index-llms-openrouter
!pip install PyPDF2

In [ ]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openrouter import OpenRouter
from llama_index.core.llms import ChatMessage
from llama_index.core.postprocessor import LongContextReorder
from llama_index.core.extractors import TitleExtractor
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import (
    StorageContext,
    Settings,
    VectorStoreIndex,
    SimpleDirectoryReader
    )

import PyPDF2
import torch
import re
import os

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
def is_scanned_pdf(pdf_path, page_sample=10, threshold=50):
    """Проверяет PDF на содержание изображений вместо текста"""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)

        checked_pages = min(page_sample, len(reader.pages))
        if checked_pages == 0:
            return False

        empty_count = 0
        for i in range(checked_pages):
            text = reader.pages[i].extract_text()
            if len(text.strip()) < threshold:
                empty_count += 1

        return empty_count == checked_pages

def pdf2txt(pdf_path):
    """Извлечение текста из PDF с помощью PyPDF2"""

    if is_scanned_pdf(pdf_path):
        print(f"Внимание: {pdf_path} похож на сканированный документ! Текст не может быть извлечен.")
        return ""

    text = ""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() + "\n\n"
    return text

def clean_for_rag(text):
    cleaned_text = re.sub(r'[^a-zA-Zа-яА-ЯёЁ0-9\s,.!?;:—]+', '', text)  # Оставляем буквы и нужные знаки
    cleaned_text = cleaned_text.strip()  # Удаляем лишние пробелы в начале и конце
    cleaned_text = re.sub(r'\n\s*\n', '\n\n', cleaned_text)
    return cleaned_text

def process_document(file_path):
    if file_path.lower().endswith('.pdf'):
        text = pdf2txt(file_path)
        if not text:
            print(f"Файл {file_path} не был обработан (причина: сканированный документ)")
            return None
    else:
        with open(file_path, 'r', encoding='utf-8-sig') as f:
            text = f.read()
    text = clean_for_rag(text)
    os.makedirs("data", exist_ok=True)
    filepath = os.path.join("data", "processed_text.txt")
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"Файл {file_path} успешно обработан")

process_document('11-0.txt')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Документ содержит отрывки из произведения "Алиса в Зазеркалье" Льюиса Кэрролла. В нем описывается приключение Алисы в мире, где все вокруг странно и нелогично. Алиса встречает различных персонажей, в том числе Короля, Королеву, Чeshire-кота, Мартовского зайца, Белого кролика и других. В тексте описываются их диалоги и действия, а также происходящие вокруг них события.

In [ ]:
rag.llm(API_KEY_RAG,"deepseek/deepseek-chat-v3-0324:free")

In [ ]:
print('Спросите меня что-нибудь о документе:\n')
while True:
    query=input('Введите вопрос: (0-завершить)\n')
    if query!='0':
        rag.ask(query)
    else:
        break

Спросите меня что-нибудь о документе:

Введите вопрос: (0-завершить)
О чем документ?
Документ представляет собой отрывки из истории о приключениях Алисы, где она взаимодействует с различными фантастическими персонажами, такими как:  
- Мышь, с которой у неё происходит неловкий разговор  
- Чеширский Кот, обсуждающий безумие и играющий в прятки  
- Король и Королева, ведущие абсурдный суд  
- Голубь, обвиняющий Алису в том, что она змея  

Основные темы:  
* Абсурдные диалоги и логические головоломки  
* Превращения и странные события  
* Конфликты с капризными персонажами  
* Игра в крокет с Королевой  

Стиль повествования — сюрреалистичная сказка с элементами нонсенса.Введите вопрос: (0-завершить)
0
